# 01. Training Notebook (MPS Safe Setup)

This notebook trains **YOLOv10n** and **YOLOv8n** on a reduced Pascal VOC setup
that stays MPS-friendly while keeping a decent accuracy/speed trade-off:

- `device = mps` when available
- moderate batch size
- larger image size
- no training plots during fitting
- explicit memory cleanup before each run

The goal is to keep the workflow simple and stable on a Mac Mini.

## What This Notebook Produces

After `Run All`, you should obtain two run folders:

- `results/mps_runs/yolov10n_main_mps_coslr_100ep_img640_b8`
- `results/mps_runs/yolov8n_main_mps_coslr_100ep_img640_b8`

Each run stores `best.pt`, `last.pt`, and `results.csv`.
The visualization notebook reads these folders directly.

In [4]:
import gc
import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO
from ultralytics import settings as ul_settings

def find_project_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'notebooks').exists():
            return candidate
    return start

ROOT = find_project_root()
os.chdir(ROOT)

RESULTS = ROOT / 'results' / 'mps_runs'
DATASETS_DIR = ROOT / 'data' / 'datasets'
RESULTS.mkdir(parents=True, exist_ok=True)
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
ul_settings.update({'datasets_dir': str(DATASETS_DIR), 'runs_dir': str(RESULTS)})

SEED = 42
DATA_YAML = 'VOC.yaml'
IMGSZ = 640
BATCH = 8
EPOCHS = 100
FRACTION = 0.10
LR0 = 0.01
LRF = 0.001
WARMUP_EPOCHS = 3.0
MOMENTUM = 0.937
WEIGHT_DECAY = 5e-4
OPTIMIZER = 'SGD'
WORKERS = 1
RUN_TAG = 'mps_coslr_100ep_img640_b8'

MODELS = {
    'YOLOv10n': 'yolov10n.pt',
    'YOLOv8n': 'yolov8n.pt',
}

print({
    'root': str(ROOT),
    'device': DEVICE,
    'imgsz': IMGSZ,
    'batch': BATCH,
    'epochs': EPOCHS,
    'fraction': FRACTION,
})

{'root': '/Users/guillaumerabeau/deep-learning-paper-YOLOv10', 'device': 'mps', 'imgsz': 640, 'batch': 8, 'epochs': 100, 'fraction': 0.1}


In [5]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.backends.mps.is_available() and hasattr(torch, 'mps') and hasattr(torch.mps, 'manual_seed'):
        torch.mps.manual_seed(seed)

def save_json(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')

def train_if_needed(model_name: str) -> dict:
    run_name = f"{Path(model_name).stem}_main_{RUN_TAG}"
    run_dir = RESULTS / run_name
    best = run_dir / 'weights' / 'best.pt'
    last = run_dir / 'weights' / 'last.pt'
    csv_path = run_dir / 'results.csv'

    if best.exists() and last.exists() and csv_path.exists():
        print(f'[skip] {run_name}')
    else:
        print(f'[train] {run_name} on {DEVICE}')
        set_seed(SEED)
        gc.collect()
        if DEVICE == 'mps' and hasattr(torch, 'mps'):
            torch.mps.empty_cache()
        model = YOLO(model_name)
        model.train(
            data=DATA_YAML,
            epochs=EPOCHS,
            fraction=FRACTION,
            imgsz=IMGSZ,
            batch=BATCH,
            device=DEVICE,
            optimizer=OPTIMIZER,
            lr0=LR0,
            lrf=LRF,
            cos_lr=True,
            warmup_epochs=WARMUP_EPOCHS,
            momentum=MOMENTUM,
            weight_decay=WEIGHT_DECAY,
            workers=WORKERS,
            cache=False,
            amp=False,
            pretrained=True,
            seed=SEED,
            deterministic=True,
            patience=100,
            project=str(RESULTS),
            name=run_name,
            exist_ok=True,
            save=True,
            plots=False,
            verbose=False,
        )

    return {{
        'run_name': run_name,
        'run_dir': str(run_dir),
        'best': str(best),
        'last': str(last),
        'results_csv': str(csv_path),
    }}

In [6]:
v10_info = train_if_needed(MODELS['YOLOv10n'])
save_json({'YOLOv10n': v10_info}, RESULTS / 'run_info_yolov10n.json')
pd.DataFrame([
    {
        'model': 'YOLOv10n',
        'run_dir': v10_info['run_dir'],
        'best_exists': Path(v10_info['best']).exists(),
        'last_exists': Path(v10_info['last']).exists(),
    }
])

[train] yolov10n_main_mps_coslr_100ep_img640_b8 on mps
New https://pypi.org/project/ultralytics/8.4.60 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.12.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=VOC.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.1, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov10n.pt, momentum=0.937, mosaic=1.0, multi_sc

TypeError: unhashable type: 'dict'

In [7]:
v8_info = train_if_needed(MODELS['YOLOv8n'])
save_json({'YOLOv8n': v8_info}, RESULTS / 'run_info_yolov8n.json')
pd.DataFrame([
    {
        'model': 'YOLOv8n',
        'run_dir': v8_info['run_dir'],
        'best_exists': Path(v8_info['best']).exists(),
        'last_exists': Path(v8_info['last']).exists(),
    }
])

[train] yolov8n_main_mps_coslr_100ep_img640_b8 on mps
New https://pypi.org/project/ultralytics/8.4.60 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.52 🚀 Python-3.12.13 torch-2.12.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=VOC.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.1, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scal

TypeError: unhashable type: 'dict'

## Next Step

Open `notebooks/02_viz_compare_yolov10_mps.ipynb` and run all cells.
That notebook compares the two saved runs directly.